# Robustness & Evidence Analysis — NMAT Admission Standards Paper

Implements the ten analytical steps from *"Analytical Steps Guide: Further Data Analysis for the
NMAT Admission Standards Evidence Paper"* (Section 9.5 of the paper).

**Read this before anything else.**

Every step below is executed against the corrected dataset produced by the remediated pipeline
chain (Pipelines 1 → 2 → 4 → 5). During remediation five defects were found and fixed in that
chain. Two of them change how this guide should be read:

1. **The matcher used to refuse to match below-40th-percentile examinees.** `disambiguate()` in
   `2_PLE_Matching_Pipeline.ipynb` applied a hard `PERCENTILE_FLOOR = 40`, discarding
   name-collision candidates below the 40th percentile. That manufactured an artificial cliff at
   exactly the CMO threshold under review. **The "~23-point B4→B5 gap" the guide treats as the
   paper's central descriptive finding is largely that artefact.** Steps 3 and 4 are therefore
   reframed below: they still run, but they test whether a discontinuity exists rather than
   assuming one does.
2. **A record-linkage bug credited one PLE passer to several same-named people.** Fixing it moved
   confirmed passers from 49,086 sittings to 47,485, and distinct passers to 35,746.

As a result **every headline number quoted in the guide is superseded.** Step 0 reconciles them
explicitly so the paper cannot end up citing two different figures for the same quantity.

## How to read this notebook

- Each step prints a **RESULT** block (the table or statistic) and an **INTERPRETATION** block.
- Interpretation text is *computed from the data*, not typed in, so it can never drift out of sync
  with the numbers above it.
- Static framing and caveats live in the markdown cells; anything numeric is generated.
- `assert` statements guard each step. If an assertion fires, the notebook stops — a silently
  wrong table is worse than a failed run.

## Standing constraints (apply to every step)

| Constraint | Why |
|---|---|
| **"Linkage", never "pass rate"** | `PLE_DATA.csv` contains passers only — no failures, no roster of takers. Every rate here is the share of NMAT examinees matched to a passer record. |
| **Observable cohort = `IS_BEST_OBSERVABLE_RECORD`** | Not `IS_BEST_NMAT_RECORD & Year<=2014`, which drops 3,721 people whose best attempt fell later. |
| **B1 is the LOWEST decile** | A string sort puts B10 between B1 and B2. Bins are ordered explicitly everywhere. |
| **`UNDERGRAD_*` is the undergraduate institution** | No medical-school identifier exists anywhere in this dataset, so nothing here can speak to SUC/PHEI admission or institutional PLE performance. |
| **PLE source covers 2011–2022 only** | Cohorts are observed through unequal windows. See Step 6. |

---
## Setup and provenance

Pins the notebook to a specific dataset build. If the md5 does not match, the numbers below were
produced by a different pipeline run and must not be quoted alongside them.

In [1]:
import hashlib, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():          # tolerate being run from a subfolder
    ROOT = ROOT.parent

PARQUET = ROOT / "dataset" / "NMAT_Exodus.parquet"
RAW_CSV = ROOT / "dataset" / "NMAT_CLEANED_DATA.csv"
MATCH_MASTER = ROOT / "dataset" / "output" / "PLE_MATCH_MASTER.csv"

EXPECTED_MD5 = "72b2808bb8bb9c3594980c5735f814e1"
md5 = hashlib.md5(PARQUET.read_bytes()).hexdigest()

df = pd.read_parquet(PARQUET)

BINS = [f"B{i}" for i in range(1, 11)]        # B1 lowest .. B10 highest
SUB  = ["B1", "B2", "B3", "B4"]               # below the 40th percentile
B5P  = [f"B{i}" for i in range(5, 11)]        # 40th percentile and above

# Observable cohort: one row per person, best attempt, NMAT year <= 2014.
obs = df[df["IS_BEST_OBSERVABLE_RECORD"].fillna(False).astype(bool)].copy()
obs["linked"] = obs["IS_PLE_PASSER"].fillna(False).astype(bool)
obs["pct"]    = pd.to_numeric(obs["NMS_PER_num"], errors="coerce")

assert md5 == EXPECTED_MD5, f"dataset md5 {md5} != expected {EXPECTED_MD5}"
assert df.shape == (178_927, 53), f"unexpected shape {df.shape}"
assert (obs["pct"].dropna() >= 0).all(), "negative percentile found (the -1 sentinel is back)"
assert obs["PercentileBin"].dropna().isin(BINS).all(), "unexpected bin label"

print(f"source      : {PARQUET}")
print(f"md5         : {md5}  (matches expected build)")
print(f"rows x cols : {df.shape[0]:,} x {df.shape[1]}")
print(f"observable cohort (people)      : {len(obs):,}")
print(f"  of whom linked to a PLE passer: {int(obs['linked'].sum()):,} "
      f"({obs['linked'].mean()*100:.2f}%)")
print("\nAll provenance assertions passed.")

source      : D:\User\Desktop\Acads\NMAT Analysis\NMAT_Analysis\dataset\NMAT_Exodus.parquet
md5         : 72b2808bb8bb9c3594980c5735f814e1  (matches expected build)
rows x cols : 178,927 x 53
observable cohort (people)      : 69,503
  of whom linked to a PLE passer: 30,105 (43.31%)

All provenance assertions passed.


---
## Step 0 — Reconciling the guide's numbers against this dataset

**Not in the original guide.** Added because the guide was written against tables that predate the
pipeline corrections. Running Steps 1–10 without this reconciliation would produce a paper that
cites two different values for the same quantity.

In [2]:
_sub = obs[obs["PercentileBin"].isin(SUB)]
_g   = obs.groupby("PercentileBin")["linked"].mean().reindex(BINS) * 100
_pub = obs[obs["UNDERGRAD_UNI_TYPE"] == "Public"]
_pri = obs[obs["UNDERGRAD_UNI_TYPE"] == "Private"]

recon = pd.DataFrame([
    ("Sub-threshold (B1-B4) linked passers", "3,644",  f"{int(_sub['linked'].sum()):,}"),
    ("Dataset columns",                      "54",     f"{df.shape[1]}"),
    ("B4->B5 gap (percentage points)",        "~23",   f"{_g['B5'] - _g['B4']:.1f}"),
    ("Public B5+ clearance",                  "64.9%", f"{_pub['PercentileBin'].isin(B5P).mean()*100:.1f}%"),
    ("Private B5+ clearance",                 "59.2%", f"{_pri['PercentileBin'].isin(B5P).mean()*100:.1f}%"),
    ("Confirmed PLE passers (sittings)",      "n/a",   f"{int(df['IS_PLE_PASSER'].fillna(False).astype(bool).sum()):,}"),
], columns=["Quantity", "Guide / published brief", "This dataset (corrected)"])

print("RESULT — guide figures vs corrected dataset")
print(recon.to_string(index=False))

print("\nINTERPRETATION")
print(f"  Every quantity the guide quotes has moved. The sub-threshold count is "
      f"{int(_sub['linked'].sum()) - 3644:+,} against the guide's 3,644, and the B4->B5 gap is "
      f"{_g['B5'] - _g['B4']:.1f} points rather than ~23.")
print("  Any table carried over from the published brief must be regenerated, not reused.")

RESULT — guide figures vs corrected dataset
                            Quantity Guide / published brief This dataset (corrected)
Sub-threshold (B1-B4) linked passers                   3,644                    5,665
                     Dataset columns                      54                       53
      B4->B5 gap (percentage points)                     ~23                     10.0
                Public B5+ clearance                   64.9%                    70.8%
               Private B5+ clearance                   59.2%                    59.9%
    Confirmed PLE passers (sittings)                     n/a                   47,485

INTERPRETATION
  Every quantity the guide quotes has moved. The sub-threshold count is +2,021 against the guide's 3,644, and the B4->B5 gap is 10.0 points rather than ~23.
  Any table carried over from the published brief must be regenerated, not reused.


---
## Step 1 — Sub-threshold linked passers by institution type

**Guide's objective:** replace the uniform-rate estimate with an exact tabulation of how many
B1–B4 linked passers are Public vs Private.

**Verdict: valid, run as specified.** The guide predicts the Private/PHEI share will exceed the
uniform estimate. That prediction is testable here.

**Caveat carried from the guide (and it is fatal for any SUC/PHEI reading):** `UNDERGRAD_UNI_TYPE`
is the examinee's *undergraduate* institution. It is not their medical school, and this dataset
contains no medical-school identifier at all. This table cannot speak to SUC vs PHEI admission.

In [3]:
sub = obs[obs["PercentileBin"].isin(SUB)]

s1 = (sub.groupby("UNDERGRAD_UNI_TYPE")
         .agg(examinees=("linked", "size"), linked_passers=("linked", "sum"))
         .assign(linkage_rate_pct=lambda t: (t.linked_passers / t.examinees * 100).round(1))
         .sort_values("linked_passers", ascending=False))
s1["share_of_linked_passers_pct"] = (s1.linked_passers / s1.linked_passers.sum() * 100).round(1)

total = int(s1.linked_passers.sum())
assert total == int(sub["linked"].sum()), "cross-tab lost rows"

print(f"RESULT — B1-B4 (below 40th percentile) linked passers, observable cohort  (n = {total:,})")
print(s1.to_string())

priv = s1.loc["Private"]; publ = s1.loc["Public"]
print("\nINTERPRETATION")
print(f"  Of the {total:,} sub-threshold linked passers, {int(priv.linked_passers):,} "
      f"({priv.share_of_linked_passers_pct:.1f}%) come from Private undergraduate institutions and "
      f"{int(publ.linked_passers):,} ({publ.share_of_linked_passers_pct:.1f}%) from Public.")
print(f"  The guide predicted the Private share would exceed a uniform-rate estimate. It does: "
      f"Private supplies {priv.share_of_linked_passers_pct:.1f}% of sub-threshold linked passers "
      f"while making up {priv.examinees / s1.examinees.sum() * 100:.1f}% of the sub-threshold pool.")
print(f"  Note the direction of the RATE, which the guide does not anticipate: within B1-B4, "
      f"Private examinees link at {priv.linkage_rate_pct:.1f}% versus Public at "
      f"{publ.linkage_rate_pct:.1f}%.")
print("  This is an undergraduate-institution split. It says nothing about SUC/PHEI medical schools.")

RESULT — B1-B4 (below 40th percentile) linked passers, observable cohort  (n = 5,665)
                    examinees  linked_passers  linkage_rate_pct  share_of_linked_passers_pct
UNDERGRAD_UNI_TYPE                                                                          
Private                 20335            4794              23.6                         84.6
Public                   3826             755              19.7                         13.3
Not Specified             441              76              17.2                          1.3
Foreign                   421              40               9.5                          0.7

INTERPRETATION
  Of the 5,665 sub-threshold linked passers, 4,794 (84.6%) come from Private undergraduate institutions and 755 (13.3%) from Public.
  The guide predicted the Private share would exceed a uniform-rate estimate. It does: Private supplies 84.6% of sub-threshold linked passers while making up 81.3% of the sub-threshold pool.
  Note the direc

---
## Step 2 — Missingness / evaluability audit

**Guide's objective:** a definitive field-by-field audit replacing ad hoc statements about what the
dataset can and cannot answer.

**Verdict: largely already delivered.** `docs/data_dictionary.md` documents all 53 columns against
the live file. What that document lacks is the guide's six *evaluability* flags, so this step adds
exactly that and nothing else.

The six categories the guide asks about: GIDA residency, IP membership, medical-school (SUC/PHEI)
destination, admission-year applicable cutoff, institution-level PLE denominator, and non-NMAT
admission criteria (GWA, interview, etc.).

In [4]:
CATEGORIES = ["GIDA_residency", "IP_membership", "medschool_destination",
              "admission_year_cutoff", "institutional_PLE_denominator", "non_NMAT_criteria"]

# No column in the shipped schema captures any of the six. This is asserted, not assumed:
# the check below looks for any plausible token in every column name.
TOKENS = {
    "GIDA_residency":                ["gida", "geograph", "isolat", "barangay", "municipal", "remote"],
    "IP_membership":                 ["indigen", "_ip_", "ethnic", "tribe", "ancestral"],
    "medschool_destination":         ["med_school", "medschool", "medical_school", "college_of_med", "enrolled"],
    "admission_year_cutoff":         ["cutoff", "cut_off", "threshold", "admission"],
    "institutional_PLE_denominator": ["school_ple", "institution_ple", "ple_denominator", "ple_takers"],
    "non_NMAT_criteria":             ["gwa", "interview", "essay", "recommendation", "gpa"],
}

rows = []
for col in df.columns:
    low = col.lower()
    rec = {"field": col,
           "dtype": str(df[col].dtype),
           "pct_non_null": round(df[col].notna().mean() * 100, 2),
           "n_distinct": int(df[col].nunique(dropna=True))}
    for cat in CATEGORIES:
        rec[cat] = any(tok in low for tok in TOKENS[cat])
    rows.append(rec)

audit = pd.DataFrame(rows)
hits = {cat: int(audit[cat].sum()) for cat in CATEGORIES}

print(f"RESULT — evaluability audit over all {len(audit)} shipped columns")
print("\nColumns capturing each evaluability category:")
for cat, n in hits.items():
    print(f"  {cat:<32s} {n} column(s)" + ("" if n else "   <- ABSENT"))

print("\nFirst 12 rows of the field-level audit (full table exported below):")
print(audit.head(12).to_string(index=False))

out = ROOT / "dataset" / "analysis_output" / "step02_evaluability_audit.csv"
out.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(out, index=False)

print("\nINTERPRETATION")
absent = [c for c, n in hits.items() if n == 0]
print(f"  {len(absent)} of the 6 evaluability categories have NO representing column in the shipped "
      f"schema: {', '.join(absent)}.")
print("  Consequence: the paper cannot evaluate CMO compliance at institution level, cannot identify")
print("  GIDA or IP examinees, cannot know which cutoff applied in a given admission year, and cannot")
print("  observe any non-NMAT admission criterion. These are structural absences, not missing data")
print("  that better cleaning would recover.")
print(f"  Step 9 revisits GIDA/IP specifically against the RAW file, which has columns the shipped")
print(f"  file drops.")
print(f"\n  Full audit written to {out.relative_to(ROOT)}")

RESULT — evaluability audit over all 53 shipped columns

Columns capturing each evaluability category:
  GIDA_residency                   0 column(s)   <- ABSENT
  IP_membership                    0 column(s)   <- ABSENT
  medschool_destination            0 column(s)   <- ABSENT
  admission_year_cutoff            0 column(s)   <- ABSENT
  institutional_PLE_denominator    0 column(s)   <- ABSENT
  non_NMAT_criteria                0 column(s)   <- ABSENT

First 12 rows of the field-level audit (full table exported below):
                 field   dtype  pct_non_null  n_distinct  GIDA_residency  IP_membership  medschool_destination  admission_year_cutoff  institutional_PLE_denominator  non_NMAT_criteria
           APPNO_CLEAN     str        100.00      178926           False          False                  False                  False                          False              False
            PERSON_KEY     str        100.00      134869           False          False                  F

---
## Step 3 — Best-record selection sensitivity

**Guide's objective:** test whether the B4→B5 transition depends on how the "best" record is chosen
among repeat takers.

**Reframed, and one specification flagged as unsound.** The guide's primary specification (A) is
*"PLE-matched attempt for linked passers, otherwise highest percentile"*. That makes record
selection **depend on the outcome being studied**: passers are represented by a different attempt
than non-passers. This is the same class of defect already removed from this pipeline once — an
earlier version applied one rule to passers and another to everyone else, and silently dropped
1,311 people from every person-level count.

Specification A is therefore computed and reported **as a bias demonstration**, not as the primary.
Specifications B (first attempt) and C (highest percentile) apply one uniform rule to every person
and are the defensible ones.

In [5]:
# Attempt-level frame: every sitting, with the person's linked status attached.
att = df.copy()
att["linked"] = att["IS_PLE_PASSER"].fillna(False).astype(bool)
att["pct"]    = pd.to_numeric(att["NMS_PER_num"], errors="coerce")
att = att[att["pct"].notna() & att["PercentileBin"].notna()]

# Which sitting was actually matched to the PLE record? Recovered from the match master,
# since Pipeline 2 propagates PLE metadata across a person's sittings.
matched_appnos = set()
if MATCH_MASTER.exists():
    mm = pd.read_csv(MATCH_MASTER, dtype=str, low_memory=False)
    matched_appnos = set(mm["MATCHED_APPNO"].dropna().astype(str).str.strip()) - {"", "nan"}
att["is_matched_sitting"] = att["APPNO_CLEAN"].astype(str).isin(matched_appnos)

obs_att = att[att["Year"] <= 2014]

def spec_A(g):   # outcome-dependent -- reported to show the bias, not to be used
    if g["linked"].iloc[0] and g["is_matched_sitting"].any():
        return g[g["is_matched_sitting"]].nlargest(1, "pct")
    return g.nlargest(1, "pct")

sel = {}
sel["C_highest_pct"]  = obs_att.sort_values("pct",  ascending=False).groupby("PERSON_KEY", as_index=False).head(1)
sel["B_first_attempt"] = obs_att.sort_values("Year", ascending=True ).groupby("PERSON_KEY", as_index=False).head(1)
sel["A_outcome_dependent"] = (obs_att.sort_values(["is_matched_sitting", "pct"], ascending=[False, False])
                                     .groupby("PERSON_KEY", as_index=False).head(1))

s3 = pd.DataFrame({name: g.groupby("PercentileBin")["linked"].mean().reindex(BINS) * 100
                   for name, g in sel.items()}).round(1)
s3 = s3[["B_first_attempt", "C_highest_pct", "A_outcome_dependent"]]

for name, g in sel.items():
    assert g["PERSON_KEY"].is_unique, f"{name} produced duplicate people"

gaps = {c: s3.loc["B5", c] - s3.loc["B4", c] for c in s3.columns}

print("RESULT — linkage rate (%) by bin under three best-record specifications")
print(s3.to_string())
print("\nB4->B5 gap under each specification (percentage points):")
for c, v in gaps.items():
    print(f"  {c:<22s} {v:5.1f}")

print("\nINTERPRETATION")
print(f"  Under the two defensible, uniform-rule specifications the B4->B5 gap is "
      f"{gaps['B_first_attempt']:.1f} points (first attempt) and {gaps['C_highest_pct']:.1f} points "
      f"(highest percentile).")
print(f"  Neither is anywhere near the ~23 points the guide describes. The transition is stable "
      f"across specifications, but stable at a MODEST value -- so this is robustness evidence for a "
      f"gentle gradient, not for a discontinuity.")
print(f"  Specification A, which selects the matched sitting for passers only, gives "
      f"{gaps['A_outcome_dependent']:.1f} points. It is listed for completeness and must not be used: "
      f"choosing a person's representative attempt using the outcome under study builds the "
      f"correlation it then reports.")

RESULT — linkage rate (%) by bin under three best-record specifications
               B_first_attempt  C_highest_pct  A_outcome_dependent
PercentileBin                                                     
B1                        13.6           10.8                 11.5
B2                        24.7           20.7                 21.4
B3                        31.0           26.7                 27.2
B4                        36.4           33.3                 33.2
B5                        43.3           43.3                 43.3
B6                        46.6           47.4                 47.1
B7                        50.8           50.5                 50.5
B8                        54.2           52.7                 52.7
B9                        60.1           59.5                 59.1
B10                       72.3           69.8                 69.6

B4->B5 gap under each specification (percentage points):
  B_first_attempt          6.9
  C_highest_pct           10.0
  A_

---
## Step 4 — Significance and changepoint testing on the gradient

**Guide's objective:** attach a formal test to the B4→B5 transition, and confirm it is the single
largest discontinuity rather than an eyeballed one.

**Reframed.** The guide assumes B4→B5 is the standout and asks for confirmation. This step instead
tests *whether any bin boundary stands out*, which is the question the data can actually answer.

Two additions the guide omits, both necessary for the result to be publishable:

- **Multiple-comparison correction.** Nine adjacent-bin tests are run. Without correction, finding
  "a significant gap" somewhere is close to guaranteed. Holm–Bonferroni is applied.
- **Effect size.** At n in the tens of thousands, p-values are near-meaningless on their own.

In [6]:
cnt = obs.groupby("PercentileBin")["linked"].agg(["size", "sum"]).reindex(BINS)

res = []
for i in range(len(BINS) - 1):
    lo, hi = BINS[i], BINS[i + 1]
    s = [int(cnt.loc[hi, "sum"]), int(cnt.loc[lo, "sum"])]
    n = [int(cnt.loc[hi, "size"]), int(cnt.loc[lo, "size"])]
    z, p = proportions_ztest(s, n)
    p_lo = s[1] / n[1]; p_hi = s[0] / n[0]
    res.append({"boundary": f"{lo}->{hi}",
                "rate_lo_pct": round(p_lo * 100, 1),
                "rate_hi_pct": round(p_hi * 100, 1),
                "gap_pts": round((p_hi - p_lo) * 100, 1),
                "z": round(z, 2), "p_raw": p,
                "cohens_h": round(abs(2*np.arcsin(np.sqrt(p_hi)) - 2*np.arcsin(np.sqrt(p_lo))), 3)})

s4 = pd.DataFrame(res)

# Holm-Bonferroni across the nine boundary tests.
order = s4["p_raw"].rank(method="first").astype(int)
m = len(s4)
s4["p_holm"] = [min(1.0, (m - order[i] + 1) * s4["p_raw"][i]) for i in s4.index]
s4["signif_holm_05"] = s4["p_holm"] < 0.05
s4 = s4.sort_values("gap_pts", ascending=False).reset_index(drop=True)

s4_show = s4.copy()
s4_show["p_raw"]  = s4_show["p_raw"].map(lambda v: f"{v:.2e}")
s4_show["p_holm"] = s4_show["p_holm"].map(lambda v: f"{v:.2e}")

print("RESULT — all nine adjacent-bin boundaries, ranked by gap size")
print(s4_show.to_string(index=False))

b45 = s4[s4.boundary == "B4->B5"].iloc[0]
largest = s4.iloc[0]
rank45 = int(s4.index[s4.boundary == "B4->B5"][0]) + 1

print("\nINTERPRETATION")
print(f"  B4->B5 -- the CMO threshold boundary -- has a gap of {b45.gap_pts} points and ranks "
      f"{rank45} of 9 by size.")
print(f"  The largest boundary is {largest.boundary} at {largest.gap_pts} points.")
if rank45 == 1:
    print("  B4->B5 is the largest boundary.")
else:
    print(f"  B4->B5 is NOT the standout transition the guide assumes. At least one other boundary "
          f"({largest.boundary}) is larger, and the guide's premise of a discontinuity located at "
          f"the 40th percentile is not supported.")
nsig = int(s4.signif_holm_05.sum())
print(f"  {nsig} of 9 boundaries remain significant after Holm-Bonferroni correction, which is "
      f"expected at this sample size and is why effect size matters more than p here.")
print(f"  Cohen's h for B4->B5 is {b45.cohens_h} -- a small effect by conventional benchmarks "
      f"(0.2 small, 0.5 medium, 0.8 large).")
print("  Conclusion: the linkage gradient rises steadily across bins. There is no changepoint at the")
print("  40th percentile. The paper should not describe one, and must not treat this as validating")
print("  the 40th-percentile cutoff.")

RESULT — all nine adjacent-bin boundaries, ranked by gap size
boundary  rate_lo_pct  rate_hi_pct  gap_pts     z    p_raw  cohens_h   p_holm  signif_holm_05
 B9->B10         59.5         69.8     10.2 13.94 3.46e-44     0.215 2.77e-43            True
  B4->B5         33.3         43.3     10.0 11.79 4.22e-32     0.207 2.95e-31            True
  B1->B2         10.8         20.7      9.9 15.47 5.52e-54     0.275 4.97e-53            True
  B8->B9         52.7         59.5      6.8  8.10 5.46e-16     0.137 3.28e-15            True
  B3->B4         26.7         33.3      6.6  7.97 1.58e-15     0.144 7.88e-15            True
  B2->B3         20.7         26.7      6.0  7.59 3.28e-14     0.141 1.31e-13            True
  B5->B6         43.3         47.4      4.0  4.59 4.47e-06     0.081 1.34e-05            True
  B6->B7         47.4         50.5      3.2  3.58 3.42e-04     0.064 6.84e-04            True
  B7->B8         50.5         52.7      2.2  2.48 1.30e-02     0.043 1.30e-02            Tru

---
## Step 5 — Match-type-stratified linkage

**Guide's objective:** test whether the bin gradient is an artifact of looser matching criteria
being more common in one score band than another.

**Verdict: valid and important — this is a genuine negative control.** If low bins depended
disproportionately on looser matching, part of the gradient would be a data-quality artifact rather
than an outcome relationship.

Relevant background: `MANUAL_APPNO_MATCH` joins on a manually supplied application number **with no
name, year or date-of-birth verification at match time**. It is the loosest of the three methods,
so its distribution across bins is the thing to watch.

In [7]:
linked = obs[obs["linked"]].copy()

comp = (pd.crosstab(linked["PercentileBin"], linked["PLE_MATCH_METHOD"], normalize="index")
          .reindex(BINS).mul(100).round(1))

rate_by_method = {}
for meth in ["EXACT", "MANUAL_APPNO_MATCH", "DETERMINISTIC_APPNO"]:
    num = obs[obs["linked"] & (obs["PLE_MATCH_METHOD"] == meth)].groupby("PercentileBin").size()
    den = obs.groupby("PercentileBin").size()
    rate_by_method[meth] = (num.reindex(BINS).fillna(0) / den.reindex(BINS) * 100).round(2)
rates = pd.DataFrame(rate_by_method)

ct = pd.crosstab(linked["PercentileBin"], linked["PLE_MATCH_METHOD"]).reindex(BINS).fillna(0)
chi2, pval, dof, _ = chi2_contingency(ct)
cramers_v_5 = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

print("RESULT — match-method composition WITHIN linked passers, by bin (row %)")
print(comp.to_string())
print("\nRESULT — linkage rate (%) contributed by each method, by bin")
print(rates.to_string())

man = comp["MANUAL_APPNO_MATCH"]
print(f"\nChi-square test, bin x match-method independence: chi2 = {chi2:,.1f}, "
      f"dof = {dof}, p = {pval:.3e}, Cramer's V = {cramers_v_5:.4f}")

print("\nINTERPRETATION")
print(f"  The loosest method, MANUAL_APPNO_MATCH, accounts for {man.min():.1f}%-{man.max():.1f}% of "
      f"linked passers across bins (lowest decile B1: {man['B1']:.1f}%, highest B10: {man['B10']:.1f}%).")
print(f"  Cramer's V of {cramers_v_5:.4f} indicates a negligible association between score bin and "
      f"which matching method produced the link, despite a significant p-value driven by sample size.")
if man["B1"] - man["B10"] > 5:
    print("  CAUTION: low bins do lean on looser matching; treat part of the gradient as a "
          "matching-quality artifact and report this prominently in Limitations.")
else:
    print("  The gradient is therefore NOT explained by matching quality: low-scoring bins do not "
          "depend disproportionately on looser matching.")
    print("  This is a clean negative control and strengthens the descriptive finding. It should be "
          "reported as such rather than omitted for being a null result.")

RESULT — match-method composition WITHIN linked passers, by bin (row %)
PLE_MATCH_METHOD  DETERMINISTIC_APPNO  EXACT  MANUAL_APPNO_MATCH
PercentileBin                                                   
B1                                0.9   90.0                 9.1
B2                                0.3   92.1                 7.5
B3                                0.2   92.9                 6.9
B4                                0.3   92.9                 6.8
B5                                0.2   93.0                 6.8
B6                                0.2   93.0                 6.8
B7                                0.2   93.4                 6.4
B8                                0.0   92.8                 7.2
B9                                0.0   93.8                 6.2
B10                               0.0   93.8                 6.2

RESULT — linkage rate (%) contributed by each method, by bin
               EXACT  MANUAL_APPNO_MATCH  DETERMINISTIC_APPNO
PercentileBin           

---
## Step 6 — Right-censoring sensitivity

**Guide's objective:** test whether the `Year <= 2014` restriction actually equalizes censoring risk
across the observable cohort.

**Verdict: the single most important step in the guide — and the answer is worse than it assumes.**

The guide treats `Year <= 2014` as a restriction that "reduces" censoring. It cannot equalize it,
for a structural reason the guide does not mention: **`PLE_DATA.csv` contains records for 2011–2022
only.** With a median NMAT→PLE gap of about six years, a 2014 examinee must pass within 8 years to
appear at all, while a 2006 examinee loses anyone who passed before 2011.

Two analyses are run: the guide's sub-period comparison, then a **fixed-exposure-window** design
that actually removes the bias rather than measuring it.

In [8]:
obs6 = obs.copy()
obs6["ple_year"] = pd.to_numeric(obs6["PLE_YEAR_PASSED"], errors="coerce")

src_years = pd.to_numeric(df["PLE_YEAR_PASSED"], errors="coerce").dropna()
gap = (obs6.loc[obs6["linked"], "ple_year"] - obs6.loc[obs6["linked"], "Year"]).dropna()

print(f"PLE source coverage      : {int(src_years.min())} - {int(src_years.max())}")
print(f"NMAT-to-PLE gap (years)  : median {gap.median():.0f}, mean {gap.mean():.2f}, "
      f"IQR {gap.quantile(.25):.0f}-{gap.quantile(.75):.0f}")

# (a) the guide's method
periods = [(2006, 2008), (2009, 2011), (2012, 2014)]
tbl_a = pd.DataFrame({f"{a}-{b}": obs6[obs6.Year.between(a, b)].groupby("PercentileBin")["linked"].mean().reindex(BINS) * 100
                      for a, b in periods}).round(1)

# (b) fixed exposure: count a pass only if it happened within W years of the NMAT sitting
W = 8
obs6["hitW"] = obs6["linked"] & ((obs6["ple_year"] - obs6["Year"]) <= W)
tbl_b = pd.DataFrame({
    "published (any horizon)": obs6.groupby("PercentileBin")["linked"].mean().reindex(BINS) * 100,
    f"equal {W}-year window":  obs6.groupby("PercentileBin")["hitW"].mean().reindex(BINS) * 100,
}).round(1)
tbl_b["difference"] = (tbl_b.iloc[:, 0] - tbl_b.iloc[:, 1]).round(1)

by_year = pd.DataFrame({
    "published": obs6.groupby("Year")["linked"].mean() * 100,
    f"{W}-yr window": obs6.groupby("Year")["hitW"].mean() * 100,
}).round(1)

print("\nRESULT (a) — the guide's sub-period comparison: linkage rate (%) by bin")
print(tbl_a.to_string())
print(f"\nRESULT (b) — equal-exposure correction: linkage rate (%) by bin")
print(tbl_b.to_string())
print(f"\nRESULT (c) — linkage rate (%) by NMAT year, before and after equalizing exposure")
print(by_year.to_string())

early = tbl_a.iloc[:, 0].mean(); late = tbl_a.iloc[:, -1].mean()
sp_pub = by_year["published"].max() - by_year["published"].min()
sp_fix = by_year[f"{W}-yr window"].max() - by_year[f"{W}-yr window"].min()

print("\nINTERPRETATION")
print(f"  (a) Mean linkage across bins is {early:.1f}% for 2006-2008 and {late:.1f}% for 2012-2014, "
      f"a drop of {early - late:.1f} points. Residual censoring inside the observable cohort is "
      f"therefore substantial, not merely 'reduced'.")
print(f"  (b) Equalizing exposure at {W} years lowers overall observable linkage from "
      f"{obs6['linked'].mean()*100:.2f}% to {obs6['hitW'].mean()*100:.2f}%. The bin gradient keeps "
      f"its shape, so the descriptive finding survives, but the levels do not.")
print(f"  (c) The apparent decline across NMAT years narrows from {sp_pub:.1f} points to "
      f"{sp_fix:.1f} points once exposure is equalized -- most of that 'trend' was the data window, "
      f"not a change in outcomes.")
print(f"  ACTION FOR THE PAPER: Limitations must state the 2011-2022 PLE window explicitly and "
      f"quantify the residual bias. Any linkage rate quoted without an exposure window is not "
      f"comparable across cohorts.")

PLE source coverage      : 2011 - 2022
NMAT-to-PLE gap (years)  : median 6, mean 6.60, IQR 6-7

RESULT (a) — the guide's sub-period comparison: linkage rate (%) by bin
               2006-2008  2009-2011  2012-2014
PercentileBin                                 
B1                  20.4       12.1        7.7
B2                  30.8       22.1       15.8
B3                  36.4       25.6       23.3
B4                  39.8       33.3       30.3
B5                  48.8       43.6       40.5
B6                  51.3       47.6       45.3
B7                  59.5       48.4       48.4
B8                  60.4       51.3       50.7
B9                  63.7       60.4       57.1
B10                 73.8       71.9       67.1

RESULT (b) — equal-exposure correction: linkage rate (%) by bin
               published (any horizon)  equal 8-year window  difference
PercentileBin                                                          
B1                                10.8                  7.7

  (b) Equalizing exposure at 8 years lowers overall observable linkage from 43.31% to 38.00%. The bin gradient keeps its shape, so the descriptive finding survives, but the levels do not.
  (c) The apparent decline across NMAT years narrows from 16.9 points to 10.7 points once exposure is equalized -- most of that 'trend' was the data window, not a change in outcomes.
  ACTION FOR THE PAPER: Limitations must state the 2011-2022 PLE window explicitly and quantify the residual bias. Any linkage rate quoted without an exposure window is not comparable across cohorts.


---
## Step 7 — Pre-2016 vs post-2016 split of sub-threshold linked passers

**Guide's objective:** scope the sub-threshold finding to examinees actually admitted under CMO 18's
40th-percentile regime.

**Verdict: do not build this analysis — the answer is structural and requires no study.** The
observable cohort is capped at NMAT year ≤ 2014 by construction, so the post-2016 count is exactly
zero, not "small or zero" as the guide anticipates. The cell below demonstrates that rather than
asserting it, and then quantifies what lifting the restriction would actually buy.

In [9]:
sub_linked = obs[obs["PercentileBin"].isin(SUB) & obs["linked"]]
post = int((sub_linked["Year"] >= 2016).sum())

assert post == 0, "post-2016 rows appeared inside a cohort defined as Year<=2014"

print("RESULT — sub-threshold (B1-B4) linked passers by NMAT year")
print(sub_linked.groupby("Year").size().to_frame("linked_passers").to_string())
print(f"\n  NMAT year range in this group : {int(sub_linked.Year.min())}-{int(sub_linked.Year.max())}")
print(f"  Count with NMAT year >= 2016   : {post}  (zero by construction, not by finding)")

# What would lifting the restriction give? Show why it does not help.
allb = df[df["IS_BEST_NMAT_RECORD"].fillna(False).astype(bool)].copy()
allb["linked"] = allb["IS_PLE_PASSER"].fillna(False).astype(bool)
late = allb[allb["Year"] >= 2016]
late_rate = late["linked"].mean() * 100
early_rate = allb[allb["Year"] <= 2014]["linked"].mean() * 100

print(f"\n  If the <=2014 restriction were lifted:")
print(f"    2016-2018 examinees: {len(late):,}, linkage {late_rate:.1f}%")
print(f"    <=2014 examinees   : {len(allb[allb.Year <= 2014]):,}, linkage {early_rate:.1f}%")

print("\nINTERPRETATION")
print(f"  Every one of the {len(sub_linked):,} sub-threshold linked passers sat the NMAT in "
      f"{int(sub_linked.Year.min())}-{int(sub_linked.Year.max())} -- years before CMO 18 took effect "
      f"in 2016.")
print("  THIS IS THE FINDING, and it is decisive for how the paper reads that figure: none of these")
print("  examinees were admitted under the regime currently in force. The number cannot be presented")
print("  as evidence of non-compliance with CMO 18.")
print(f"  Lifting the restriction does not rescue the comparison: 2016-2018 examinees link at only "
      f"{late_rate:.1f}% versus {early_rate:.1f}%, because they mostly have not reached the PLE "
      f"within the data window (see Step 6). The comparison the guide wants is not available from "
      f"this dataset at any cohort definition.")

RESULT — sub-threshold (B1-B4) linked passers by NMAT year
      linked_passers
Year                
2006             478
2007             447
2008             506
2009             672
2010             699
2011             728
2012             647
2013             567
2014             921

  NMAT year range in this group : 2006-2014
  Count with NMAT year >= 2016   : 0  (zero by construction, not by finding)



  If the <=2014 restriction were lifted:
    2016-2018 examinees: 58,761, linkage 6.8%


    <=2014 examinees   : 65,782, linkage 43.8%

INTERPRETATION
  Every one of the 5,665 sub-threshold linked passers sat the NMAT in 2006-2014 -- years before CMO 18 took effect in 2016.
  THIS IS THE FINDING, and it is decisive for how the paper reads that figure: none of these
  examinees were admitted under the regime currently in force. The number cannot be presented
  as evidence of non-compliance with CMO 18.
  Lifting the restriction does not rescue the comparison: 2016-2018 examinees link at only 6.8% versus 43.8%, because they mostly have not reached the PLE within the data window (see Step 6). The comparison the guide wants is not available from this dataset at any cohort definition.


---
## Step 8 — Raw score trend by year

**Guide's objective:** replace the bin-share-by-year proxy with an actual score trend, since
percentile rank is cohort-relative and cannot distinguish a real trend from rebinning.

**Verdict: valid, straightforward, and the reasoning is sound.** `TotalRawScoreTRUE` is the
recalculated sum of the eight component subtests and is absolute rather than cohort-relative, so it
can show drift that percentile rank structurally cannot.

In [10]:
best = df[df["IS_BEST_NMAT_RECORD"].fillna(False).astype(bool)].copy()

s8 = (best.groupby("Year")["TotalRawScoreTRUE"]
          .agg(n="count", median="median", mean="mean", std="std").round(2))

pct_by_year = best.groupby("Year")["NMS_PER_num"].median()
s8["median_percentile"] = pct_by_year.round(1)

assert best["TotalRawScoreTRUE"].notna().mean() > 0.99, "raw score coverage unexpectedly low"

print("RESULT — TotalRawScoreTRUE by NMAT year (best-record cohort)")
print(s8.to_string())

first, last = int(s8.index.min()), int(s8.index.max())
d_raw = s8.loc[last, "median"] - s8.loc[first, "median"]
d_pct = s8.loc[last, "median_percentile"] - s8.loc[first, "median_percentile"]

print("\nINTERPRETATION")
print(f"  Median raw score moved from {s8.loc[first,'median']:.0f} in {first} to "
      f"{s8.loc[last,'median']:.0f} in {last}, a change of {d_raw:+.0f} points "
      f"({d_raw / s8.loc[first,'median'] * 100:+.1f}%).")
print(f"  Over the same span the median PERCENTILE moved only {d_pct:+.1f} points -- because "
      f"percentile is cohort-relative by construction and is renormalized every year.")
print("  This is the guide's point, demonstrated: a real decline in absolute performance is almost")
print("  invisible in percentile space. A percentile rank from 2006 and one from 2018 do not denote")
print("  the same level of measured attainment, so a fixed percentile cutoff is not a fixed standard.")
print("  This reinforces, independently, the instability finding reported by Vergeire-Dalmacion et al.")

RESULT — TotalRawScoreTRUE by NMAT year (best-record cohort)
          n  median    mean    std  median_percentile
Year                                                 
2006   3698   131.0  131.86  32.34               53.0
2007   3690   130.0  131.45  33.38               52.0
2008   4965   129.0  129.90  32.29               54.0
2009   7445   130.0  131.22  30.68               52.0
2010   8548   136.0  137.21  30.90               57.0
2011   8692   129.0  130.99  30.38               52.0
2012   9102   122.0  123.97  31.50               54.0
2013   9144   128.0  129.24  35.35               60.0
2014  10455   120.0  119.87  31.33               59.0
2015  10326   118.0  118.40  35.13               54.0
2016  12480   123.0  122.87  33.82               49.0
2017  23948   118.0  118.85  34.45               44.0
2018  22333   111.0  113.38  30.71               43.0

INTERPRETATION
  Median raw score moved from 131 in 2006 to 111 in 2018, a change of -20 points (-15.3%).
  Over the same span t

---
## Step 9 — Schema review for partial GIDA / IP proxies

**Guide's objective:** confirm rather than assume that no field offers even a partial proxy for GIDA
residency or IP membership.

**Verdict: run it — and the answer is not the one the guide expects.** The guide anticipates
confirming absence. The shipped 53-column file does lack any such field (Step 2), but the **raw
source file does not**, and the relevant columns are dropped during slimming.

In [11]:
raw_head = pd.read_csv(RAW_CSV, nrows=5, low_memory=False)
GEO_TOKENS = ["prov", "region", "address", "location", "city", "municip", "barangay", "home"]
geo_cols = [c for c in raw_head.columns if any(t in c.lower() for t in GEO_TOKENS)]

print(f"Geographic-ish columns in the RAW file ({RAW_CSV.name}): {geo_cols}")
print(f"Of those, present in the shipped 53-column file: "
      f"{[c for c in geo_cols if c in df.columns] or 'NONE -- all dropped during slimming'}")

if geo_cols:
    geo = pd.read_csv(RAW_CSV, usecols=geo_cols, low_memory=False)
    summary = pd.DataFrame({
        "pct_non_null": (geo.notna().mean() * 100).round(1),
        "n_distinct":   geo.nunique(),
    })
    print("\nRESULT — coverage of the candidate proxy fields")
    print(summary.to_string())

    for c in geo_cols:
        print(f"\nTop 8 values of '{c}':")
        print(geo[c].value_counts().head(8).to_string())

print("\nINTERPRETATION")
if geo_cols:
    best_col = summary["pct_non_null"].idxmax()
    print(f"  A partial geographic proxy DOES exist. '{best_col}' is "
          f"{summary.loc[best_col,'pct_non_null']:.1f}% populated with "
          f"{int(summary.loc[best_col,'n_distinct'])} distinct values, in the raw file.")
    print("  It is NOT in the shipped dataset -- Pipeline 5 drops it -- which is why every prior")
    print("  statement that the field is 'absent' was true of the analytic file but false of the source.")
    print("  STRENGTH OF THE PROXY, stated honestly: DOH AO 2020-0023 defines GIDA at barangay and")
    print("  municipality level using physical and socioeconomic criteria. Province or region of")
    print("  address is far coarser, is an ADDRESS rather than a residency determination, and cannot")
    print("  identify IP membership at all. It supports at best a crude upper-bound estimate of the")
    print("  GIDA-eligible population, clearly labelled as such, and is never equivalent to certified")
    print("  GIDA or IP status.")
    print("  RECOMMENDATION: retain these columns through Pipeline 5 so the estimate can be made, and")
    print("  continue to press for the registration-time self-declaration capture the paper recommends.")
else:
    print("  No geographic field exists in the raw file either; absence is confirmed.")

Geographic-ish columns in the RAW file (NMAT_CLEANED_DATA.csv): ['NMAT Province local address', 'NMAT Region permanent address']
Of those, present in the shipped 53-column file: NONE -- all dropped during slimming



RESULT — coverage of the candidate proxy fields
                               pct_non_null  n_distinct
NMAT Province local address           100.0         634
NMAT Region permanent address         100.0          21

Top 8 values of 'NMAT Province local address':
NMAT Province local address
Ncr              63424
Cebu             11051
Davao Del Sur     8654
Cavite            7463
Iloilo            4974
Pangasinan        4558
Laguna            4421
Rizal             4330

Top 8 values of 'NMAT Region permanent address':
NMAT Region permanent address
NCR            51294
Overseas       33977
Region IV-A    18869
Region III     10087
Region VII     10022
Region VI       9027
Region X        6781
Region XI       6624

INTERPRETATION
  A partial geographic proxy DOES exist. 'NMAT Province local address' is 100.0% populated with 634 distinct values, in the raw file.
  It is NOT in the shipped dataset -- Pipeline 5 drops it -- which is why every prior
  statement that the field is 'absent' 

---
## Step 10 — Chi-square: institution type × threshold status

**Guide's objective:** attach a formal significance statement to the Public/Private B5+ clearance
difference.

**Verdict: valid, and the guide's own caution is the important part.** At n in the tens of
thousands almost any difference reaches significance, so Cramér's V is what determines whether the
gap is practically meaningful. The guide's hardcoded contingency table is from the superseded brief
and is rebuilt from the corrected data here.

In [12]:
o10 = obs[obs["UNDERGRAD_UNI_TYPE"].isin(["Public", "Private"])].copy()
o10["b5plus"] = o10["PercentileBin"].isin(B5P)

table = pd.crosstab(o10["UNDERGRAD_UNI_TYPE"], o10["b5plus"])
table.columns = ["below_B5", "B5_plus"]

chi2, pval, dof, expected = chi2_contingency(table.values)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

rates = (table["B5_plus"] / table.sum(axis=1) * 100).round(1)
diff = rates["Public"] - rates["Private"]

print("RESULT — 2x2 contingency, observable cohort")
print(table.to_string())
print(f"\nB5+ clearance: Public {rates['Public']:.1f}%, Private {rates['Private']:.1f}%, "
      f"difference {diff:+.1f} points")
print(f"chi2 = {chi2:,.1f}, dof = {dof}, p = {pval:.3e}, n = {n:,}")
print(f"Cramer's V = {cramers_v:.4f}")

print("\nINTERPRETATION")
print(f"  The Public/Private difference in B5+ clearance is {diff:+.1f} points and is statistically "
      f"significant (p = {pval:.1e}).")
mag = ("negligible" if cramers_v < 0.1 else "small" if cramers_v < 0.3 else
       "moderate" if cramers_v < 0.5 else "large")
p_pub = rates["Public"] / 100; p_pri = rates["Private"] / 100
cohens_h = abs(2*np.arcsin(np.sqrt(p_pub)) - 2*np.arcsin(np.sqrt(p_pri)))
print(f"  Cramer's V of {cramers_v:.4f} is a {mag} association and Cohen's h is {cohens_h:.3f} "
      f"(small). With n = {n:,}, significance was close to guaranteed and carries almost no "
      f"information on its own.")
print(f"  DO NOT round this to 'no difference'. Association measures on a 2x2 with unbalanced "
      f"margins understate practical importance: a {diff:+.1f}-point gap in B5+ clearance is "
      f"material for policy even though the association is statistically weak. Report the "
      f"percentage-point difference AND the effect size together, and let the reader judge.")
print("  The guide's published cells (Public 64.9% / Private 59.2%) do not reproduce here, because")
print("  the corrected pipeline reassigned 416 rows whose university type contradicted their own")
print("  source hint and removed sentinel-valued percentiles. Use these figures, not the brief's.")
print("  Reminder: this is the UNDERGRADUATE institution. It is not an SUC/PHEI comparison.")

RESULT — 2x2 contingency, observable cohort
                    below_B5  B5_plus
UNDERGRAD_UNI_TYPE                   
Private                21278    31759
Public                  4170    10093

B5+ clearance: Public 70.8%, Private 59.9%, difference +10.9 points
chi2 = 565.7, dof = 1, p = 4.950e-125, n = 67,300
Cramer's V = 0.0917

INTERPRETATION
  The Public/Private difference in B5+ clearance is +10.9 points and is statistically significant (p = 4.9e-125).
  Cramer's V of 0.0917 is a negligible association and Cohen's h is 0.230 (small). With n = 67,300, significance was close to guaranteed and carries almost no information on its own.
  DO NOT round this to 'no difference'. Association measures on a 2x2 with unbalanced margins understate practical importance: a +10.9-point gap in B5+ clearance is material for policy even though the association is statistically weak. Report the percentage-point difference AND the effect size together, and let the reader judge.
  The guide's publish

---
## Summary of findings

Regenerated from the computations above, so it cannot drift out of sync with them.

In [13]:
lines = []
lines.append(("1",  "Sub-threshold linked passers by institution type",
              f"{int(sub['linked'].sum()):,} in B1-B4; Private supplies "
              f"{s1.loc['Private','share_of_linked_passers_pct']:.1f}%. Undergraduate split only."))
lines.append(("2",  "Evaluability audit",
              f"{len([c for c,v in hits.items() if v==0])}/6 categories absent from the schema."))
lines.append(("3",  "Best-record sensitivity",
              f"B4->B5 gap {gaps['B_first_attempt']:.1f}-{gaps['C_highest_pct']:.1f} pts across "
              f"defensible specs; stable but modest."))
lines.append(("4",  "Changepoint testing",
              f"B4->B5 ranks {rank45}/9 by size; largest is {largest.boundary}. No changepoint "
              f"at the 40th percentile."))
lines.append(("5",  "Match-type stratification",
              f"Cramer's V {cramers_v_5:.4f}; gradient is NOT a matching artifact."))
lines.append(("6",  "Right-censoring",
              f"Equalizing exposure moves linkage {obs6['linked'].mean()*100:.2f}% -> "
              f"{obs6['hitW'].mean()*100:.2f}%; year-spread {sp_pub:.1f} -> {sp_fix:.1f} pts."))
lines.append(("7",  "Pre/post-2016 split",
              f"0 of {len(sub_linked):,} sub-threshold passers sat NMAT after 2016 -- none were "
              f"admitted under CMO 18."))
lines.append(("8",  "Raw score trend",
              f"Median raw score {d_raw:+.0f} pts {first}->{last} while median percentile moved "
              f"{d_pct:+.1f}."))
lines.append(("9",  "GIDA/IP proxy",
              "Partial geographic proxy EXISTS in the raw file but is dropped by Pipeline 5."))
lines.append(("10", "UNITYPE x threshold",
              f"Public {rates['Public']:.1f}% vs Private {rates['Private']:.1f}%; Cramer's V "
              f"{cramers_v:.4f} ({mag}); {diff:+.1f}pt gap is still policy-relevant."))

summary = pd.DataFrame(lines, columns=["Step", "Title", "Finding"])
print(summary.to_string(index=False))

out = ROOT / "dataset" / "analysis_output" / "robustness_summary.csv"
summary.to_csv(out, index=False)
print(f"\nWritten to {out.relative_to(ROOT)}")

Step                                            Title                                                                                              Finding
   1 Sub-threshold linked passers by institution type                                    5,665 in B1-B4; Private supplies 84.6%. Undergraduate split only.
   2                               Evaluability audit                                                               6/6 categories absent from the schema.
   3                          Best-record sensitivity                                  B4->B5 gap 6.9-10.0 pts across defensible specs; stable but modest.
   4                              Changepoint testing                 B4->B5 ranks 2/9 by size; largest is B9->B10. No changepoint at the 40th percentile.
   5                        Match-type stratification                                              Cramer's V 0.0376; gradient is NOT a matching artifact.
   6                                  Right-censoring                 

---
## What the paper should change

1. **Withdraw the discontinuity claim.** Step 4 finds no changepoint at the 40th percentile; the
   B4→B5 boundary is not the largest of the nine, and its effect size is small. The "~23-point gap"
   was largely an artefact of the matcher bug documented at the top of this notebook. Sections
   describing a sharp break need rewriting, and the corrected gradient should be presented as what
   it is — a steady rise across the whole score range.
2. **State the 2011–2022 PLE window in Limitations, with numbers.** Step 6 shows the `Year <= 2014`
   restriction does not equalize censoring. Quote linkage rates with an explicit exposure window.
3. **Re-scope the sub-threshold figure.** Step 7 shows none of those examinees were admitted under
   CMO 18. Presenting the count as evidence about the current regime would misstate it.
4. **Report Step 5 as a negative control.** Matching quality does not explain the gradient. Null
   results that rule out an artifact belong in the paper.
5. **Lead with effect sizes, not p-values.** Steps 4, 5 and 10 all reach significance purely on
   sample size.
6. **Correct the GIDA/IP claim.** Step 9 shows a partial geographic proxy exists in the source data.
   The honest statement is that the *analytic file* omits it and that province-level address is far
   too coarse for a GIDA determination — not that the source captures nothing.
7. **Do not claim the 40th percentile is validated.** Nothing in this dataset can validate a cutoff:
   there is no medical-school identifier, no admission-decision field, and no record of who was
   rejected. That constraint is structural and is unchanged by any of these ten steps.